# Flexible segmentation training

Interactive front-end for the `segmentation` package. Pick an **architecture**,
override any hyperparameters you want, train, and inspect the results — all
without editing source files.

Every run is tracked to MLflow (`sqlite:///mlflow.db`, experiment
`urban-segmentation`) and the best model is registered in the MLflow Model
Registry as `cityvision-segmentation-<arch>`.

> Run the cells top to bottom. Edit the **Configure** cell to change the run.

## 1. Setup — make the `segmentation` package importable

In [ ]:
import os
import sys

# Walk up from the notebook until we find the folder that contains the
# `segmentation` package, then put it on sys.path. Works no matter where the
# notebook is launched from.
_d = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_d, "segmentation")) and os.path.dirname(_d) != _d:
    _d = os.path.dirname(_d)
PROJECT_ROOT = _d
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

In [ ]:
import torch

from segmentation.config import ARCH_PRESETS, make_config
from segmentation.models import ARCHITECTURES, build_model
from segmentation.engine import fit

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("Available architectures:", ", ".join(sorted(ARCHITECTURES)))

## 2. Browse the architecture presets

These are the defaults applied for each `--arch`. Anything you set in the
**Configure** cell overrides them.

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.DataFrame(ARCH_PRESETS).T

## 3. Configure the run

Set `ARCH`, then edit any knobs in `OVERRIDES`. Leave a value as `None`
(or remove the line) to keep the architecture's preset default.

In [ ]:
# ----- pick the architecture of the solution -----
ARCH = "resnet50"            # one of: unet, resnet34, resnet50, segnet, vgg

# ----- override any hyperparameters (None = use the preset default) -----
OVERRIDES = dict(
    epochs=None,            # int   — number of epochs
    batch_size=None,        # int   — batch size (lower if you hit OOM)
    lr=None,                # float — base (decoder) learning rate
    encoder_lr_mult=None,   # float — encoder LR = lr * this (pretrained models)
    patience=None,          # int   — early-stopping patience (0 disables)
    dropout=None,           # float — U-Net only
    img_size=None,          # tuple — (H, W), e.g. (512, 1024)
    scheduler=None,         # None or "plateau"
    pretrained=None,        # bool  — ImageNet encoder weights
    balanced=None,          # bool  — class-balanced sampler
    run_name=None,          # str   — MLflow run name
    registered_model_name=None,  # str — MLflow Model Registry name
)

# Build the config: preset for ARCH + your overrides.
config = make_config(ARCH, **OVERRIDES)

# Show the resolved configuration.
from dataclasses import asdict
for k, v in asdict(config).items():
    print(f"  {k:24s}: {v}")

### (Optional) quick CPU smoke test

Uncomment to force a tiny, fast run that proves the pipeline works end-to-end
(useful when you have no GPU).

In [ ]:
# config = make_config(ARCH, epochs=1, batch_size=1, img_size=(256, 512))
# print("Smoke-test config:", config.arch, config.epochs, config.batch_size, config.img_size)

## 4. Train

Runs the shared engine: data loading, training/validation loop, per-class IoU
logging, best-checkpoint saving, and MLflow registration. Returns the best
validation mIoU.

In [ ]:
best_val_miou = fit(config)
print(f"\nBest val mIoU: {best_val_miou:.4f}")
print(f"Checkpoint:    {config.checkpoint_path}")
print(f"Registered as: {config.registered_model_name}")

## 5. Load the trained model back

Either from the local checkpoint, or from the MLflow Model Registry (latest
version of `cityvision-segmentation-<arch>`).

In [ ]:
from segmentation.data import NUM_CLASSES

device = "cuda" if torch.cuda.is_available() else "cpu"

# Option A — rebuild the architecture and load the best checkpoint weights.
model = build_model(config.arch, num_classes=NUM_CLASSES,
                    pretrained=False, dropout=config.dropout).to(device)
model.load_state_dict(torch.load(config.checkpoint_path, map_location=device))
model.eval()

# Sanity check the output shape on a dummy input.
with torch.no_grad():
    h, w = config.img_size
    out = model(torch.randn(1, 3, h, w, device=device))
print("Output shape:", tuple(out.shape), "(expected (1, %d, %d, %d))" % (NUM_CLASSES, h, w))

In [ ]:
# Option B — load the latest version from the MLflow Model Registry.
# import mlflow
# import mlflow.pytorch
# mlflow.set_tracking_uri(f"sqlite:///{PROJECT_ROOT}/mlflow.db")
# registry_model = mlflow.pytorch.load_model(f"models:/{config.registered_model_name}/latest")
# registry_model.eval()
# print("Loaded from registry:", config.registered_model_name)

## 6. (Optional) Compare several architectures

Train a list of architectures back-to-back and collect their best scores.

In [ ]:
# ARCHES_TO_COMPARE = ["unet", "resnet34", "resnet50"]
# results = {}
# for a in ARCHES_TO_COMPARE:
#     cfg = make_config(a, epochs=5)          # short runs for comparison
#     results[a] = fit(cfg)
# import pandas as pd
# pd.Series(results, name="best_val_mIoU").sort_values(ascending=False)